In [213]:
import time
import os

import porespy as ps
import numpy as np
import scipy as sc

os.chdir("..")
%run ./pyflowsolver/volumeManager.py
%run ./pyflowsolver/sparseArray.py
%run ./pyflowsolver/fastLaplacian.py
%run ./pyflowsolver/darcySolver.py
os.chdir("notebooks")

In [215]:
import pandas as pd

pores_table=pd.read_pickle("/home/romcenci/Desktop/pore.pd")
throats_table=pd.read_pickle("/home/romcenci/Desktop/throat.pd")

matrix = np.zeros((len(pores_table),len(pores_table)))
vector = np.zeros(len(pores_table))
for i in range(len(throats_table)):
    conn0 = throats_table["throat.conns_0"][i]
    conn1 = throats_table["throat.conns_1"][i]
    cond = 2.0*throats_table["throat.sub_conductivity"][i]
    if ~np.isnan(pores_table["pore.bc.value"][conn0]):
        vector[conn1] -= cond
        matrix[conn1][conn1] -= cond
    elif ~np.isnan(pores_table["pore.bc.value"][conn1]):
        vector[conn0] -= cond
        matrix[conn0][conn0] -= cond
    else:
        matrix[conn0][conn1] = cond
        matrix[conn1][conn0] = cond
        matrix[conn0][conn0] -= cond
        matrix[conn1][conn1] -= cond
        
matrix = matrix[np.isnan(pores_table["pore.bc.value"])][:,np.isnan(pores_table["pore.bc.value"])]
vector = vector[np.isnan(pores_table["pore.bc.value"])]

In [216]:
SIZE = 20
vol = ps.generators.blobs(shape=(SIZE, SIZE, SIZE), blobiness=0.4, porosity=0.55)
vol, n_lab = sc.ndimage.label(vol)
vol = (vol==1)
if vol.sum() == 0:
    print('error')
else:
    print('image OK')

cond_vol = (vol==1)*100 #porosity map ndarray uint8 0..100
cond_vol = fast_laplacian_volume_generator(
    cond_vol, 
    (1., 1., 1.), 
    )

volume_manager = VolumeManager(cond_vol)

image OK


In [217]:
solver = DarcySolver()
sparse_A, sparse_b = volume_manager.get_sparse_system_from_dense(matrix, vector)

In [218]:
sparse_b = vector

In [219]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_cg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        sparse_b,
        max_iterations=max_iterations, # sqrt(n) for n x n system
        target_error=1.0e-10, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
#raveled_solution = volume_manager.ravel_sparse_solution(solution)
solution

array([ 9.99998990e-01,  4.90730177e-09,  1.62524208e-30,  2.54666413e-07,
        9.99999991e-01,  4.94639841e-06,  9.99999985e-01,  1.00000001e+00,
        1.00000009e+00,  1.00000000e+00,  1.00000000e+00,  1.00000012e+00,
        1.00000020e+00,  1.00007636e+00,  1.00000002e+00,  1.00000033e+00,
        9.99999855e-01,  1.00000004e+00,  1.00000005e+00,  1.00000004e+00,
        9.93801929e-09,  5.09894523e-02,  2.94389126e-13,  1.31352345e-07,
        9.99978754e-01,  1.20793387e-21,  2.43410336e-07,  2.27912565e-08,
        2.06036313e-12,  1.92531766e-09,  3.77624282e-09,  9.99978692e-01,
       -1.91205961e-09,  8.33901086e-11,  2.88605627e-13,  1.64430775e-08,
        1.00000007e+00,  9.99999986e-01,  1.07714984e-09,  2.57169889e-09,
        6.13602142e-14,  8.92364757e-08,  1.31003902e-06,  1.24215268e-20,
        9.61595963e-07,  3.19282373e-08,  2.51626003e-06,  8.79948800e-08,
        9.99999976e-01,  9.69084472e-07,  7.71471633e-09,  6.61247740e-06,
        9.99999945e-01,  

In [212]:
from scipy import stats
stats.describe(solution)

DescribeResult(nobs=309, minmax=(-0.00043306347530836447, 1.0045209900151062), mean=0.13278440344545647, variance=0.11197346685606331, skewness=2.188296167889745, kurtosis=2.81813232977509)